# Meshes with different BCs

In [ ]:
import sys
with open("./../../../PATHS.txt") as file:
  paths = file.read().splitlines()
sys.path.extend(paths)

In [ ]:
from dd_nm_rom import env
env.set(
  backend="numpy",
  device="cpu",
  device_idx=0,
  nb_threads=4,
  epsilon=1e-10,
  floatx="float64",
  seed=0
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from matplotlib import cm

In [ ]:
from dd_nm_rom import fom as fom_mod
from dd_nm_rom import field as field_mod
from dd_nm_rom.elements import mesh as mesh_mod

In [ ]:
from dd_nm_rom.postproc.utils import set_style
plt = set_style(plt)
with_title = False

In [ ]:
# DD Mesh
nx_intr = 4
ny_intr = 4
lx_sub = 0.5
ly_sub = 0.5
x0 = 0.0
y0 = 0.0
n_sub_x = 2
n_sub_y = 2
# PDE
viscosity = 1e-3
bc_type = "periodic"

In [ ]:
mesh = mesh_mod.MeshDD(
  nx_intr=nx_intr,
  ny_intr=ny_intr,
  lx_sub=lx_sub,
  ly_sub=ly_sub,
  x0=x0,
  y0=y0,
  n_sub_x=n_sub_x,
  n_sub_y=n_sub_y,
  with_bounds=True if bc_type == "periodic" else False
)
mesh.build()

field = field_mod.SinPeak(mesh=mesh, mu_lim=[0.9,1.1], bc_type=bc_type)
field.set_params(mu=field.sample_design_space())

fom = fom_mod.Burgers2D(
  nu=viscosity,
  mesh=mesh
)
fom.build(field)

dd_fom = fom_mod.DDBurgers2D(fom)
dd_fom.build()

In [ ]:
path_to_figs = "/g/g92/zanardi1/Workspace/Codes/DD-NM-ROM/run/unsteady/meshes"
if not with_title:
  path_to_figs += "_notitle"
os.makedirs(path_to_figs, exist_ok=True)

In [ ]:
m = ["o", "^", "d", "P", ">", "*", "1", "2", "3", "4", "8"]
size = 300

In [ ]:
X, Y = mesh.grid
xx, yy = X.flatten(), Y.flatten()

# plot interior states
plt.figure(figsize=(7,7))
for i, s in enumerate(dd_fom.subdomains):
  indices = s.elem_states["interior"].nodes_state
  plt.scatter(xx[indices], yy[indices], marker=m[i%len(m)], s=size)
plt.xlim(mesh.phylim["x"])
plt.ylim(mesh.phylim["y"])
# plt.tick_params(left = False, right = False , labelleft = False ,
#                 labelbottom = False, bottom = False)
# plt.legend()
if with_title:
  lt.title("Interior States")
filename = path_to_figs + "/interior.png"
plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# plot interface states
plt.figure(figsize=(7,7))
for i, s in enumerate(dd_fom.subdomains):
  indices = s.elem_states["interface"].nodes_state
  plt.scatter(xx[indices], yy[indices], s=size, marker=m[i%len(m)], label=f"$\Omega_{i}$")
plt.xlim(mesh.phylim["x"])
plt.ylim(mesh.phylim["y"])
# plt.tick_params(left = False, right = False , labelleft = False ,
#                 labelbottom = False, bottom = False)
# plt.legend()
if with_title:
  plt.title("Interface States")
filename = path_to_figs + "/interface.png"
plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# plot ports
plt.figure(figsize=(7,7))
for i, port in dd_fom.dd_indices.port_to_nodes.items():
  plt.scatter(xx[port], yy[port], s=size, marker=m[i%len(m)], label=f"{i}")
plt.xlim(mesh.phylim["x"])
plt.ylim(mesh.phylim["y"])
# plt.tick_params(left = False, right = False , labelleft = False ,
#                 labelbottom = False, bottom = False)
if with_title:
  plt.title("Port States")
filename = path_to_figs + "/ports.png"
plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# plot ports
plt.figure(figsize=(7,7))
for i, p in enumerate(dd_fom.dd_indices.orient_to_ports["vert"]):
  nodes = dd_fom.dd_indices.port_to_nodes[p]
  print("Port", p)
  print("> i", nodes)
  print("> x", xx[nodes])
  print("> y", yy[nodes])
  plt.scatter(xx[nodes], yy[nodes], s=size, marker=m[i%len(m)], label=f"{p}")
plt.xlim(mesh.phylim["x"])
plt.ylim(mesh.phylim["y"])
# plt.legend()
# plt.tick_params(left = False, right = False , labelleft = False ,
#                 labelbottom = False, bottom = False)
if with_title:
  plt.title("Vertical Port States")
filename = path_to_figs + "/ports_vert.png"
plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# plot ports
plt.figure(figsize=(7,7))
for i, p in enumerate(dd_fom.dd_indices.orient_to_ports["horiz"]):
  nodes = dd_fom.dd_indices.port_to_nodes[p]
  print("Port", p)
  print("> i", nodes)
  print("> x", xx[nodes])
  print("> y", yy[nodes])
  plt.scatter(xx[nodes], yy[nodes], s=size, marker=m[i%len(m)], label=f"{p}")
plt.xlim(mesh.phylim["x"])
plt.ylim(mesh.phylim["y"])
# plt.legend()
# plt.tick_params(left = False, right = False , labelleft = False ,
#                 labelbottom = False, bottom = False)
if with_title:
  plt.title("Horizontal Port States")
filename = path_to_figs + "/ports_horiz.png"
plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# plot ports
plt.figure(figsize=(7,7))
for i, p in enumerate(dd_fom.dd_indices.orient_to_ports["inner"]):
  nodes = dd_fom.dd_indices.port_to_nodes[p]
  plt.scatter(xx[nodes], yy[nodes], s=size, marker=m[i%len(m)], label=f"{i}")
plt.xlim(mesh.phylim["x"])
plt.ylim(mesh.phylim["y"])
# plt.tick_params(left = False, right = False , labelleft = False ,
#                 labelbottom = False, bottom = False)
if with_title:
  plt.title("Inner Port States")
filename = path_to_figs + "/ports_inner.png"
plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# plot residual indices
plt.figure(figsize=(7,7))
for i, s in enumerate(dd_fom.subdomains):
  indices = s.elem_states["res"].nodes_res
  plt.scatter(xx[indices], yy[indices], s=size, marker="o", label=f"$\Omega_{i}$")
plt.xlim(mesh.phylim["x"])
plt.ylim(mesh.phylim["y"])
# plt.tick_params(left = False, right = False , labelleft = False ,
#                 labelbottom = False, bottom = False)
if with_title:
  plt.title("Residuals States")
filename = path_to_figs + "/residuals.png"
plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# plot subdomain indices
plt.figure(figsize=(10,10))
colors = ["blue", "orange", "green", "red"]
for i, s in enumerate(dd_fom.subdomains):
  interior = s.elem_states["interior"].nodes_state
  interface = s.elem_states["interface"].nodes_state
  xx_i = np.concatenate((xx[interior], xx[interface]))
  yy_i = np.concatenate((yy[interior], yy[interface]))
  plt.scatter(xx_i, yy_i, s=size, marker=m[i%len(m)], label=f"$\Omega_{i}$")

plt.xlim(mesh.phylim["x"])
plt.ylim(mesh.phylim["y"])
# plt.tick_params(left = False, right = False , labelleft = False ,
#                 labelbottom = False, bottom = False)
filename = path_to_figs + "/sates.png"
plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# checks that constraint matrices are computed correctly
c = np.zeros(dd_fom.n_constraints)
vec = np.random.rand(2*mesh.nxy)
for s in dd_fom.subdomains:
  interface = s.elem_states["interface"].nodes_state
  c += s.cmat["interface"]@np.concatenate([vec[interface],vec[mesh.nxy+interface]])
print("||sum(A[i] x[i])||=", np.linalg.norm(c))

In [ ]:
# plot sparsity of jacobian
_, jac = dd_fom.res_jac(np.random.rand(dd_fom.get_ndof()))[:2]
plt.figure(figsize=(10, 10))
plt.spy(jac)
plt.show()

In [ ]:
# plot sparsity of constraint matrix
plt.figure(figsize=(10, 10))
plt.spy(dd_fom.subdomains[0].cmat["interface"])
plt.show()